# E1 · El costo del prompt en SAM — reproducir en Colab

Ejecuta el pipeline completo del Entregable 1. Con GPU (T4) el paso más caro —calcular los embeddings del encoder— toma minutos en vez de horas; el resto corre sobre la caché.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo → GPU.

In [ ]:
!git clone https://github.com/<usuario>/sam-prompt-cost.git
%cd sam-prompt-cost
# Colab ya trae torch/numpy/scipy/sklearn; solo falta el código oficial de SAM (commit fijado)
!pip -q install git+https://github.com/facebookresearch/segment-anything.git@dca509fe793f601edb92606367a655c15ac00fdf

## 1. Pesos oficiales de SAM ViT-B
Se verifica el MD5 (`01ec64d2…`), cuyo prefijo da nombre al archivo oficial.

In [ ]:
!mkdir -p checkpoints
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O checkpoints/sam_vit_b_01ec64.pth
!md5sum checkpoints/sam_vit_b_01ec64.pth

## 2. Kvasir-SEG oficial y particiones
Términos: solo investigación y educación; citar Jha et al. (MMM 2020). Si el enlace cambia, descargar desde https://datasets.simula.no/kvasir-seg/.

In [ ]:
!wget -q https://datasets.simula.no/downloads/kvasir-seg.zip -O kvasir-seg.zip
!python scripts/00_prepare_kvasir.py --official_zip kvasir-seg.zip

## 3. Fase A — teardown

In [ ]:
!python scripts/01_teardown.py
!python scripts/04_receptive_field.py

## 4. Embeddings (una pasada del encoder por imagen)

In [ ]:
!python scripts/02_encode.py --split test --precision fp32
!python scripts/02_encode.py --split dev --precision fp32
!python scripts/02_encode.py --split train400 --precision bf16

## 5. Línea base: el costo del prompt

In [ ]:
!python scripts/03_prompt_cost.py --split test --modes noprompt,point,box,clicks
!python scripts/03_prompt_cost.py --split test --modes amg --pps 16
!python scripts/05_features.py
!python scripts/05b_attention_stats.py

## 6. Probe lineal, auto-prompts e intervención (tokens aprendidos)

In [ ]:
!python scripts/06_probe.py
for k in (4, 1):
    for s in (0, 1, 2):
        !python scripts/07_learned_prompts.py --k {k} --seed {s} --init meanpoint --epochs 6 --lr 5e-3 --bs 8

## 7. Resumen y figuras

In [ ]:
!python scripts/07b_twostage.py
!python scripts/08b_subgroups.py test
!python scripts/05b_attention_stats.py
!python scripts/08_summarize.py
!python scripts/09_figures.py